# Phase 1 — Data Ingestion
## FCM Gradational Lithofacies Classification
**Author:** Kumar Yuvraj (23GG5PE02) | IIT Kharagpur

This notebook:
1. Loads all FORCE 2020 wells from `./data/raw/` — handles both `.csv` and `.las` formats
2. Concatenates into a single master DataFrame
3. Prints a data quality report (per-well NaN %, depth range, class counts)
4. Saves `./outputs/data/force2020_master.csv`
5. Previews the Raniganj Vsh reference dataset

**Checkpoint before Phase 2:** verify `force2020_master.csv` exists, NaN % is printed, and the Raniganj Vsh column name is confirmed.

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import lasio

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_DIR      = './data/raw/'
RANIGANJ_DIR = './data/raniganj/'
OUT_DIR      = './outputs/data/'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs('./outputs/figures/', exist_ok=True)

# ── Lithology code → label mapping (FORCE 2020 competition standard) ──────
# WHY: The competition encodes rock types as 5-digit integers.
# We map them to readable strings for all downstream plots and ML outputs.
LITHO_MAP = {
    30000: 'Sandstone',
    65030: 'Sandstone/Shale',
    65000: 'Shale',
    80000: 'Marl',
    74000: 'Dolomite',
    70000: 'Limestone',
    88000: 'Chalk',
    86000: 'Halite',
    99000: 'Anhydrite',
    90000: 'Tuff',
    93000: 'Coal',
    10000: 'Basement',
}

# Standard 5 logs used by the LightGBM electrofacies project
REQUIRED_LOGS = ['GR', 'RHOB', 'NPHI', 'DTC', 'RDEP']
LITH_COL      = 'FORCE_2020_LITHOFACIES_LITHOLOGY'

print('Paths OK.')
print(f'  Looking for wells in: {os.path.abspath(RAW_DIR)}')

Paths OK.
  Looking for wells in: D:\Projects\fcm-lithofacies\notebooks\data\raw


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1A — Loader functions for both CSV and LAS formats
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
os.chdir('D:/Projects/fcm-lithofacies')   # set project root

# Ensure all output directories exist
os.makedirs('outputs/data',    exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

print("Working dir:", os.getcwd())

def load_csv_well(path: str) -> pd.DataFrame | None:
    """
    WHY: The FORCE 2020 train.csv uses ';' as separator (European convention).
    Some per-well exports may use ',' — we try both.
    """
    for sep in [';', ',']:
        try:
            df = pd.read_csv(path, sep=sep, low_memory=False)
            if 'WELL' in df.columns and len(df.columns) > 3:
                return df
        except Exception:
            pass
    return None


def load_las_well(path: str) -> pd.DataFrame | None:
    """
    WHY: .las (Log ASCII Standard) is the industry-standard wire-line format.
    lasio reads it into a DataFrame where columns = curve mnemonics.
    We read the WELL name from the header and add it as a column.
    """
    try:
        las = lasio.read(path)
        df  = las.df().reset_index()  # index = DEPTH
        # Rename depth column consistently
        df.rename(columns={df.columns[0]: 'DEPTH_MD'}, inplace=True)
        # Get well name from LAS header
        well_name = las.well.WELL.value if hasattr(las.well, 'WELL') else \
                    os.path.splitext(os.path.basename(path))[0]
        df.insert(0, 'WELL', well_name)
        return df
    except Exception as e:
        print(f'  WARNING: Could not read {path}: {e}')
        return None


def standardise_columns(df: pd.DataFrame, well_name: str = None) -> pd.DataFrame:
    """
    WHY: Across CSV and LAS sources, column names can differ slightly
    (e.g. 'GR' vs 'GR_EDTC', 'DEPTH' vs 'DEPTH_MD').  This function
    finds the best match for each required column and renames it.
    """
    # Add well column if missing
    if 'WELL' not in df.columns:
        df.insert(0, 'WELL', well_name or 'UNKNOWN')

    # Standardise depth column
    for cand in ['DEPTH_MD', 'DEPT', 'DEPTH', 'MD']:
        if cand in df.columns and 'DEPTH_MD' not in df.columns:
            df.rename(columns={cand: 'DEPTH_MD'}, inplace=True)
            break

    # Map lithology codes if raw column is present
    if LITH_COL in df.columns:
        df['LITH_LABEL'] = df[LITH_COL].map(LITHO_MAP)
    elif 'LITH_LABEL' not in df.columns:
        df['LITH_LABEL'] = np.nan

    return df


print('Loader functions defined.')

Working dir: D:\Projects\fcm-lithofacies
Loader functions defined.


In [3]:
import os

# Force working directory to project root so ./data/train.csv resolves correctly
os.chdir('D:/Projects/fcm-lithofacies')
print("Working directory set to:", os.getcwd())
print("train.csv visible:", os.path.isfile('./data/train.csv'))

Working directory set to: D:\Projects\fcm-lithofacies
train.csv visible: True


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1B — Discover and load all wells from ./data/raw/
# Handles both train.csv (single-file all-wells) and per-well .las files
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

frames = []

# Check for the single-file train.csv (most common FORCE 2020 distribution)
single_csv = './data/train.csv'
if os.path.isfile(single_csv):
    print(f'Found single-file train.csv → loading …')
    df_main = load_csv_well(single_csv)
    if df_main is not None:
        df_main = standardise_columns(df_main)
        frames.append(df_main)
        print(f'  Loaded: {len(df_main):,} rows, {df_main["WELL"].nunique()} wells')

# Also scan ./data/raw/ for any per-well files
csv_files = glob.glob(os.path.join(RAW_DIR, '*.csv'))
las_files = glob.glob(os.path.join(RAW_DIR, '*.las')) + \
            glob.glob(os.path.join(RAW_DIR, '*.LAS'))

print(f'\nFound in {RAW_DIR}:  {len(csv_files)} CSV,  {len(las_files)} LAS')

for path in csv_files:
    df = load_csv_well(path)
    if df is not None:
        df = standardise_columns(df)
        frames.append(df)

for path in las_files:
    df = load_las_well(path)
    if df is not None:
        df = standardise_columns(df)
        frames.append(df)

if not frames:
    raise FileNotFoundError(
        'No data found!  Place train.csv in ./data/ or per-well files in '
        './data/raw/  (see data/README.md for download link)'
    )

# Concatenate and keep only required columns
master = pd.concat(frames, ignore_index=True, sort=False)

# Select and reorder to canonical column set
keep_cols  = ['WELL', 'DEPTH_MD'] + REQUIRED_LOGS + ['LITH_LABEL']
avail_cols = [c for c in keep_cols if c in master.columns]
master     = master[avail_cols].copy()

print(f'\nMaster DataFrame: {len(master):,} rows × {len(master.columns)} cols')
print(f'Wells: {master["WELL"].nunique()}')
print(f'Columns: {list(master.columns)}')
master.head(3)

Found single-file train.csv → loading …
  Loaded: 1,170,511 rows, 98 wells

Found in ./data/raw/:  0 CSV,  0 LAS

Master DataFrame: 1,170,511 rows × 8 cols
Wells: 98
Columns: ['WELL', 'DEPTH_MD', 'GR', 'RHOB', 'NPHI', 'DTC', 'RDEP', 'LITH_LABEL']


,WELL,DEPTH_MD,GR,RHOB,NPHI,DTC,RDEP,LITH_LABEL
0,15/9-13,494.528,80.200851,1.884186,NaN,161.131180,1.798681,Shale
1,15/9-13,494.680,79.262886,1.889794,NaN,160.603470,1.795641,Shale
2,15/9-13,494.832,74.821999,1.896523,NaN,160.173615,1.800733,Shale


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1C — Data Quality Report
# Per-well: row count, % NaN per log, depth range, unique lithofacies
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

log_cols  = [c for c in REQUIRED_LOGS if c in master.columns]
rows      = []

for well, wdf in master.groupby('WELL'):
    row = {'WELL': well, 'N_rows': len(wdf)}
    if 'DEPTH_MD' in wdf.columns:
        row['Depth_min'] = round(wdf['DEPTH_MD'].min(), 1)
        row['Depth_max'] = round(wdf['DEPTH_MD'].max(), 1)
    for log in log_cols:
        nan_pct = 100.0 * wdf[log].isna().sum() / len(wdf)
        row[f'{log}_NaN%'] = round(nan_pct, 1)
    if 'LITH_LABEL' in wdf.columns:
        row['Litho_classes'] = wdf['LITH_LABEL'].nunique()
        row['Dominant_lith'] = wdf['LITH_LABEL'].mode().iloc[0] \
                               if not wdf['LITH_LABEL'].isna().all() else 'N/A'
    rows.append(row)

qc_df = pd.DataFrame(rows).set_index('WELL')

print('=' * 70)
print('DATA QUALITY REPORT — FORCE 2020')
print('=' * 70)
print(qc_df.to_string())
print()

# ── Summary stats ─────────────────────────────────────────────────────────
print('Overall NaN % per log (across all wells):')
for log in log_cols:
    nan_pct = 100.0 * master[log].isna().sum() / len(master)
    flag = '  ← WORST' if nan_pct == max(
        100.0 * master[l].isna().sum() / len(master) for l in log_cols) else ''
    print(f'  {log:<6s}: {nan_pct:.2f}%{flag}')

print()
if 'LITH_LABEL' in master.columns:
    print('Class distribution (all wells):')
    print(master['LITH_LABEL'].value_counts().to_string())

DATA QUALITY REPORT — FORCE 2020
             N_rows  Depth_min  Depth_max  GR_NaN%  RHOB_NaN%  NPHI_NaN%  DTC_NaN%  RDEP_NaN%  Litho_classes    Dominant_lith
WELL                                                                                                                         
15/9-13       18270      494.5     3272.0      0.0        0.0       23.2       0.4        0.0              8            Shale
15/9-15       17717      485.3     3200.1      0.0        1.1       24.7       0.1        0.0              6            Shale
15/9-17       17350      472.4     3114.5      0.0        3.8       20.5       0.1        0.0              7            Shale
16/1-2         1734     2645.6     2909.0      0.0        5.5        5.1       0.0        0.0              5        Sandstone
16/1-6 A       3623     1176.0     1726.5      0.0        4.0        0.2       3.9        0.0              4            Shale
16/10-1       17675      439.4     3125.9      0.0       36.2       35.3       0.1   

In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1D — Save master CSV
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

out_path = os.path.join(OUT_DIR, 'force2020_master.csv')
master.to_csv(out_path, index=False)

print(f'✓ Saved: {out_path}')
print(f'  Shape : {master.shape}')
print(f'  Size  : {os.path.getsize(out_path) / 1e6:.1f} MB')

# ── Phase 1 Checkpoint ────────────────────────────────────────────────────
assert os.path.isfile(out_path), 'force2020_master.csv not saved!'
assert all(c in master.columns for c in log_cols), \
    f'Missing log columns: {[c for c in log_cols if c not in master.columns]}'

print()
print('✓ CHECKPOINT: force2020_master.csv exists with all 5 log columns')
print('✓ CHECKPOINT: NaN % per log printed above — note worst offender')
print('→ Now run CELL 2 to confirm the Raniganj join key')

✓ Saved: ./outputs/data/force2020_master.csv
  Shape : (1170511, 8)
  Size  : 100.3 MB

✓ CHECKPOINT: force2020_master.csv exists with all 5 log columns
✓ CHECKPOINT: NaN % per log printed above — note worst offender
→ Now run CELL 2 to confirm the Raniganj join key


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Load Raniganj Vsh reference data
# Reconstructed from GPS-mapped outcrop lithologs, IIT KGP field camps 2024
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY we reconstruct inline rather than loading a CSV:
# The pseudo-GR Vsh is derived from first-hand field litholog observations
# using the GR proxy lookup (Sand=30, Shale=120 API). This keeps the
# project self-contained — no external Raniganj CSV is needed.

GR_PROXY = {
    'Sandstone':          30,
    'Siltstone':          70,
    'Carbonaceous shale': 100,
    'Shale':              120,
    'Mudstone':           110,
    'Coal':               20,
}
GR_CLEAN, GR_SHALE = 30, 120

# ── Raw field lithologs (Ramnagar, Shunuri, Duburdi) ─────────────────────
lithologs = {
    'Ramnagar_Colliery': [
        (0.0, 'Sandstone'),  (1.5, 'Sandstone'), (3.0, 'Shale'),
        (4.5, 'Carbonaceous shale'), (6.0, 'Coal'), (8.0, 'Carbonaceous shale'),
        (9.0, 'Shale'), (10.5, 'Sandstone'), (12.0, 'Sandstone'),
        (13.5, 'Sandstone'), (15.0, 'Siltstone'), (16.5, 'Sandstone'),
        (18.0, 'Siltstone'),
    ],
    'Shunuri_Village': [
        (0.0, 'Sandstone'), (1.5, 'Siltstone'), (3.0, 'Siltstone'),
        (4.5, 'Sandstone'), (7.5, 'Sandstone'), (9.0, 'Siltstone'),
        (10.5, 'Mudstone'),
    ],
    'Duburdi_Section': [
        (0.0, 'Sandstone'), (2.0, 'Siltstone'), (3.5, 'Shale'),
        (5.0, 'Carbonaceous shale'), (6.5, 'Coal'),
        (9.5, 'Carbonaceous shale'), (10.5, 'Shale'),
    ],
}

def compute_vsh_pseudogr(litholog_entries):
    rows = []
    for depth, lith in litholog_entries:
        gr  = GR_PROXY.get(lith, 65)
        vsh = float(np.clip((gr - GR_CLEAN) / (GR_SHALE - GR_CLEAN), 0, 1))
        rows.append({
            'depth_m':      depth,
            'lithology':    lith,
            'GR_proxy':     gr,
            'VSH_PSEUDOGR': vsh,           # ← NOTE: this is the Vsh column name
            'is_reservoir': int(vsh <= 0.5)
        })
    return pd.DataFrame(rows)

raniganj_frames = {}
for section, entries in lithologs.items():
    raniganj_frames[section] = compute_vsh_pseudogr(entries)

# Combine into a single DataFrame with a SECTION column
raniganj_df = pd.concat(
    [df.assign(SECTION=sec) for sec, df in raniganj_frames.items()],
    ignore_index=True
)[['SECTION', 'depth_m', 'lithology', 'GR_proxy', 'VSH_PSEUDOGR', 'is_reservoir']]

# Save for Phase 5
raniganj_path = os.path.join(RANIGANJ_DIR if os.path.isdir(RANIGANJ_DIR)
                              else OUT_DIR, 'raniganj_vsh.csv')
os.makedirs(os.path.dirname(raniganj_path), exist_ok=True)
raniganj_df.to_csv(raniganj_path, index=False)

print('=' * 60)
print('RANIGANJ PSEUDO-GR Vsh REFERENCE DATASET')
print('=' * 60)
print(f'Columns : {list(raniganj_df.columns)}')
print(f'Sections: {raniganj_df["SECTION"].unique()}')
print(f'depth_m range: {raniganj_df["depth_m"].min():.1f} – {raniganj_df["depth_m"].max():.1f} m')
print(f'VSH_PSEUDOGR range: {raniganj_df["VSH_PSEUDOGR"].min():.3f} – {raniganj_df["VSH_PSEUDOGR"].max():.3f}')
print()
print('First 5 rows:')
print(raniganj_df.head().to_string(index=False))
print()
print('─' * 60)
print(f'✓ CHECKPOINT: Join key for Phase 5 = "VSH_PSEUDOGR" column')
print(f'  Depth key : "depth_m"  |  Section key: "SECTION"')
print(f'  Saved to  : {raniganj_path}')

RANIGANJ PSEUDO-GR Vsh REFERENCE DATASET
Columns : ['SECTION', 'depth_m', 'lithology', 'GR_proxy', 'VSH_PSEUDOGR', 'is_reservoir']
Sections: ['Ramnagar_Colliery' 'Shunuri_Village' 'Duburdi_Section']
depth_m range: 0.0 – 18.0 m
VSH_PSEUDOGR range: 0.000 – 1.000

First 5 rows:
          SECTION  depth_m          lithology  GR_proxy  VSH_PSEUDOGR  is_reservoir
Ramnagar_Colliery      0.0          Sandstone        30      0.000000             1
Ramnagar_Colliery      1.5          Sandstone        30      0.000000             1
Ramnagar_Colliery      3.0              Shale       120      1.000000             0
Ramnagar_Colliery      4.5 Carbonaceous shale       100      0.777778             0
Ramnagar_Colliery      6.0               Coal        20      0.000000             1

────────────────────────────────────────────────────────────
✓ CHECKPOINT: Join key for Phase 5 = "VSH_PSEUDOGR" column
  Depth key : "depth_m"  |  Section key: "SECTION"
  Saved to  : ./outputs/data/raniganj_vsh.csv
